# AI-Based Odia Regional Signboard System — Demo

This notebook runs entirely in Google Colab's cloud environment — **nothing is installed on your own computer**. Just run every cell in order (Runtime → Run all).

Pipeline: degraded Odia signboard image → perspective correction & restoration → Odia OCR (with confidence) → English translation.

(Text-to-speech is intentionally omitted from this project's scope.)

## 1. Get the code
If opened directly from GitHub via Colab, this cell clones the repo so the `src/` modules are available.

In [ ]:
import os
if not os.path.exists('odia-signboard-system'):
    !git clone https://github.com/Sonalip12/odia-signboard-system.git
%cd odia-signboard-system

## 2. Install dependencies (runs in Colab's cloud VM only)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-ori fonts-noto-core
!pip install -q -r requirements.txt

## 3. Generate the synthetic degraded Odia-signboard dataset

In [ ]:
%cd src
!python3 generate_dataset.py --n_per_class 6
%cd ..

## 4. Look at a sample degraded signboard

In [ ]:
from PIL import Image
import pandas as pd

labels = pd.read_csv('dataset/labels.csv')
display(labels.head())

sample_row = labels.iloc[2]
img = Image.open(f"dataset/images/{sample_row['filename']}")
print('Ground truth Odia text:', sample_row['odia_text_gt'])
print('Ground truth English:', sample_row['english_gt'])
print('Degradation level:', sample_row['degradation_level'])
img

## 5. Run the full pipeline on that sample (restoration → OCR → translation)

In [ ]:
import sys, cv2
sys.path.insert(0, 'src')
from pipeline import run_pipeline
from restoration import restore

img_path = f"dataset/images/{sample_row['filename']}"
result = run_pipeline(img_path, save_restored='outputs/sample_restored.jpg')
for k, v in result.items():
    print(f'{k}: {v}')

print('\nBefore (degraded) vs After (restored):')
display(Image.open(img_path))
display(Image.open('outputs/sample_restored.jpg'))

## 6. Full evaluation over the whole dataset

In [ ]:
%cd src
!python3 evaluate.py
%cd ..

## 7. Try your own degraded Odia signboard image
Upload an image using the Colab file-upload widget below, then run the pipeline on it.

In [ ]:
from google.colab import files
uploaded = files.upload()
fname = list(uploaded.keys())[0]
result = run_pipeline(fname, save_restored='outputs/uploaded_restored.jpg')
for k, v in result.items():
    print(f'{k}: {v}')
display(Image.open('outputs/uploaded_restored.jpg'))